# NOTE: api key is deleted from the code to git push

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = ""

In [2]:
import os
import re
import time
import pandas as pd
from google import genai
from google.genai import types
from google.genai.errors import APIError
from tqdm import tqdm



# Configs

In [28]:
INPUT_CSV = "responses_long.csv"
TERM_COLUMN = "word_clean"       #use "original_word" if i want raw responses
EXCLUDE_CONTENT_WORDS = True     #drops rows where is_content_word == True

#models
MODELS = [
    {
        "name": "gemini",
        "model_id": "gemini-3.5-flash-lite",  
        "api_key": "GEMINI_API_KEY",
        "sleep_sec": 4.0, 
        "output_csv": "vad_scores_gemini.csv",
    }
]

# Prompt setup

In [26]:

PROMPTS = {
    "valence": (
        "Could you please rate the valence of the following one or multi-word expressions, "
        "in terms of VAD model. Use a scale from -1 to 1, where -1 means very negative valence "
        "and 1 means very positive valence. Examples of words that would get a rating of -1 are "
        "pedophile, AIDS, and wreck. Examples of words that would get a rating of 1 are vacation, "
        "fantastic, and laugh. The expression is: {term}. Only answer a number from -1 to 1, "
        "allowing 3 digits after decimal points. Please limit your answer to numbers."
    ),
    "arousal": (
        "Could you please rate the arousal of the following one or multi-word expressions, "
        "in terms of VAD model. Use a scale from -1 to 1, where -1 means very calm, relaxed "
        "and 1 means very aroused, energized. Examples of words that would get a rating of -1 "
        "are grain, dull, and rest. Examples of words that would get a rating of 1 are gun, "
        "lover, and thrill. The expression is: {term}. Only answer a number from -1 to 1, "
        "allowing 3 digits after decimal points. Please limit your answer to numbers."
    ),
    "dominance": (
        "Could you please rate the dominance of the following one or multi-word expressions, "
        "in terms of VAD model. Use a scale from -1 to 1, where -1 means very weak, controlled, "
        "or submissive and 1 means very powerful, dominant, or in control. Examples of words that "
        "would get a rating of -1 are weak, frail, and helpless. Examples of words that would get "
        "a rating of 1 are powerful, leader, and king. The expression is: {term}. Only answer a "
        "number from -1 to 1, allowing 3 digits after decimal points. Please limit your answer to numbers."
    ),
}

# helper functions

In [27]:

def parse_score(text: str | None) -> float | None:
    """Extract the first valid float in [-1.0, 1.0] from model output."""
    if not text:
        return None
    match = re.search(r"-?(?:1(?:\.0+)?|0(?:\.\d+)?|\.\d+)", text.strip())
    if not match:
        return None
    val = float(match.group(0))
    if -1.0 <= val <= 1.0:
        return round(val, 3)
    return None


def query_score(
    client: genai.Client,
    model_id: str,
    dimension: str,
    term: str,
    max_retries: int = 5,
) -> float | None:
    """Query Gemini natively for a single VAD dimension with retries."""
    prompt = PROMPTS[dimension].format(term=term)
    config = types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=64,
    )

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model_id,
                contents=prompt,
                config=config,
            )
            return parse_score(response.text)
        except APIError as e:
            code = getattr(e, "code", None)
            msg = getattr(e, "message", str(e))
            # Don't retry client 4xx errors other than 429 rate limits
            if code is not None and 400 <= code < 500 and code != 429:
                print(f"\n[API {code}] {model_id} on '{term}' ({dimension}): {msg}")
                return None
            wait = 2 ** (attempt + 1)
            print(f"\n[Warning] {model_id} ({code}) on '{term}' ({dimension}): {msg}. Retrying in {wait}s...")
            time.sleep(wait)
        except Exception as e:
            print(f"\n[Error] Unexpected error on '{term}' ({dimension}): {e}")
            return None

    return None


def load_unique_terms(csv_path: str) -> list[str]:
    """Load and filter unique terms from responses_long.csv."""
    df = pd.read_csv(csv_path)

    if "is_nonempty" in df.columns:
        df = df[df["is_nonempty"].astype(bool)]

    if EXCLUDE_CONTENT_WORDS and "is_content_word" in df.columns:
        df = df[~df["is_content_word"].astype(bool)]

    terms = (
        df[TERM_COLUMN]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s != ""]
        .unique()
        .tolist()
    )
    return sorted(terms)


# MAIN

In [30]:
def run_scoring():
    terms = load_unique_terms(INPUT_CSV)
    print(f"Loaded {len(terms)} unique terms from {INPUT_CSV}.")

    for cfg in MODELS:
        api_key = os.getenv(cfg["api_key"])
        if not api_key:
            print(f"Skipping {cfg['name']}: {cfg['api_key']} not set.")
            continue

        print(f"\n--- Scoring with {cfg['name']} ({cfg['model_id']}) ---")
        client = genai.Client(api_key=api_key)
        out_path = cfg["output_csv"]

        if os.path.exists(out_path):
            existing_df = pd.read_csv(out_path)
            done_terms = set(existing_df["term"].astype(str))
            results = existing_df.to_dict("records")
            print(f"Resuming: {len(done_terms)} terms already scored.")
        else:
            done_terms = set()
            results = []

        remaining = [t for t in terms if t not in done_terms]

        try:
            for term in tqdm(remaining, desc=cfg["name"]):
                row = {"term": term}
                for dim in ("valence", "arousal", "dominance"):
                    row[dim] = query_score(client, cfg["model_id"], dim, term)
                    time.sleep(cfg["sleep_sec"])

                results.append(row)

                pd.DataFrame(results, columns=["term", "valence", "arousal", "dominance"]).to_csv(
                    out_path, index=False
                )
        finally:
            client.close()

        print(f"Saved {len(results)} scored terms to {out_path}.")


if __name__ == "__main__":
    run_scoring()

Loaded 571 unique terms from responses_long.csv.

--- Scoring with gemini (gemini-3.5-flash-lite) ---


gemini:   1%|          | 6/571 [02:59<4:30:24, 28.72s/it]


[Warning] gemini-3.5-flash-lite (503) on 'afflictive' (arousal): This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.. Retrying in 2s...


gemini:  13%|█▎        | 74/571 [25:22<2:35:59, 18.83s/it]


[Warning] gemini-3.5-flash-lite (503) on 'cartoon' (arousal): The service is currently unavailable.. Retrying in 2s...


gemini:  15%|█▌        | 88/571 [29:09<2:17:14, 17.05s/it]

... (33 outputs left out)

gemini:  32%|███▏      | 180/571 [1:33:28<21:33:29, 198.49s/it]


[Warning] gemini-3.5-flash-lite (429) on 'expressive' (valence): You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 500, model: gemini-3.5-flash-lite, description: Generate content requests per day limit exceeded on Free Tier. See https://aistudio.google.com/docs/rate-limits.
Please retry in 11h24m9.869623819s.. Retrying in 2s...

[Warning] gemini-3.5-flash-lite (429) on 'expressive' (valence): You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://aistudio.google.com/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. 
* Quota exceeded for metric: generativelanguage.googleapis.com/generate_c

gemini:  32%|███▏      | 180/571 [1:34:34<3:25:25, 31.52s/it]  


KeyboardInterrupt: 

# recovering from reaching API limit

In [22]:
# check how many rows have all three VAD scores present in the currewnt output
outputdf = pd.read_csv("/Users/yukat/Developer/voice_eval/analysis/vad_scores_gemini.csv")
print(outputdf.notna().all(axis=1).sum())

259


In [37]:

csv_path = "/Users/yukat/Developer/voice_eval/analysis/vad_scores_gemini.csv"
df = pd.read_csv(csv_path)

# Keep only rows where all 3 VAD scores exist
df_clean = df.dropna(subset=["valence", "arousal", "dominance"])
df_clean.to_csv(csv_path, index=False)

print(f"Saved {len(df_clean)} complete terms.")


Saved 516 complete terms.


# MAIN
AQ.

Ab8RN6LKb_dOMJ3nI4z9iRYG5p4poIi7OK6gifzgHkooxaOa0g
# PHOTO
AQ.

Ab8RN6LrMrf7WeIhrAaCZHh8upMM-KSNwvqeXIipo94VcS600A

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = "INSERT MY GEMINI API KEY HEREEEE"

In [32]:

from ast import For


PROMPT = (
        "Could you please rate the valence, arousal, and dominance of the following one or multi-word expressions, "
        "in terms of VAD model? For each dimension, use a scale from -1 to 1."
        
        "For valence, -1 means very negative valence and 1 means very positive valence."
        "Examples of words that would get a rating of -1 are pedophile, AIDS, and wreck."
        "Examples of words that would get a rating of 1 are vacation, fantastic, and laugh. "
    
        "For arousal, -1 means very calm, relaxed and 1 means very aroused, energized. "
        "Examples of words that would get a rating of -1 are grain, dull, and rest. "
        "Examples of words that would get a rating of 1 are gun, lover, and thrill. "

       "For dominance, -1 means very weak, controlled, or submissive and 1 means very powerful, dominant, or in control. "
       "Examples of words that would get a rating of -1 are weak, frail, and helpless. "
       "Examples of words that would get a rating of 1 are powerful, leader, and king. "
       
       "The expression is: {term}. "
       "Only answer a number from -1 to 1, allowing 3 digits after decimal points. Please limit your answer to numbers."
    
)

In [33]:
import json
import os
import time
import pandas as pd
from tqdm import tqdm
from google import genai
from google.genai import types
from google.genai.errors import APIError

INPUT_CSV = "responses_long.csv"
TERM_COLUMN = "word_clean"       # use "original_word" if you want raw responses
EXCLUDE_CONTENT_WORDS = True     # drops rows where is_content_word == True

MODELS = [
    {
        "name": "gemini",
        "model_id": "gemini-3.5-flash-lite",
        "api_key": "GEMINI_API_KEY",
        "sleep_sec": 4.0,
        "output_csv": "vad_scores_gemini.csv",
    }
]


def parse_scores(text: str | None) -> dict[str, float | None]:
    """Parse valence, arousal, and dominance floats in [-1.0, 1.0] from JSON output."""
    empty = {"valence": None, "arousal": None, "dominance": None}
    if not text:
        return empty
    try:
        data = json.loads(text.strip())
    except json.JSONDecodeError:
        return empty

    if isinstance(data, list) and len(data) > 0:
        data = data[0]
    if not isinstance(data, dict):
        return empty

    data = {k.lower(): v for k, v in data.items()}

    scores = {}
    for dim in ("valence", "arousal", "dominance"):
        try:
            val = float(data[dim])
            scores[dim] = round(val, 3) if -1.0 <= val <= 1.0 else None
        except (KeyError, TypeError, ValueError):
            scores[dim] = None
    return scores



RAW_LOG_FILE = "vad_na_raw_log.jsonl"

def log_na_raw(term: str, raw_output: str | None) -> None:
    """Append the term and raw LLM output to a JSONL log file."""
    with open(RAW_LOG_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps({"term": term, "raw_output": raw_output}, ensure_ascii=False) + "\n")


def query_scores(
    client: genai.Client,
    model_id: str,
    term: str,
    max_retries: int = 5,
) -> dict[str, float | None]:
    """Query Gemini once for all 3 VAD dimensions with retries."""
    prompt = PROMPT.format(term=term)
    config = types.GenerateContentConfig(
        temperature=0.0,
        max_output_tokens=128,
        response_mime_type="application/json",
    )

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=model_id,
                contents=prompt,
                config=config,
            )
            scores = parse_scores(response.text)
            if any(v is None for v in scores.values()):
                print(f"\n[NA logged] '{term}' -> {response.text!r}")
                log_na_raw(term, response.text)
            return scores
        except APIError as e:
            code = getattr(e, "code", None)
            msg = getattr(e, "message", str(e))
            if code == 429 and ("per day" in msg.lower() or "quota" in msg.lower()):
                raise RuntimeError("DAILY_QUOTA_EXCEEDED") from e
            if code is not None and 400 <= code < 500 and code != 429:
                print(f"\n[API {code}] {model_id} on '{term}': {msg}")
                log_na_raw(term, f"APIError {code}: {msg}")
                return {"valence": None, "arousal": None, "dominance": None}
            wait = 2 ** (attempt + 1)
            print(f"\n[Warning] {model_id} ({code}) on '{term}': {msg}. Retrying in {wait}s...")
            time.sleep(wait)
        except Exception as e:
            print(f"\n[Error] Unexpected error on '{term}': {e}")
            log_na_raw(term, f"Exception: {e}")
            return {"valence": None, "arousal": None, "dominance": None}

    log_na_raw(term, "MAX_RETRIES_EXCEEDED")
    return {"valence": None, "arousal": None, "dominance": None}



def load_unique_terms(csv_path: str) -> list[str]:
    """Load and filter unique terms from responses_long.csv."""
    df = pd.read_csv(csv_path)

    if "is_nonempty" in df.columns:
        df = df[df["is_nonempty"].astype(bool)]

    if EXCLUDE_CONTENT_WORDS and "is_content_word" in df.columns:
        df = df[~df["is_content_word"].astype(bool)]

    terms = (
        df[TERM_COLUMN]
        .dropna()
        .astype(str)
        .str.strip()
        .loc[lambda s: s != ""]
        .unique()
        .tolist()
    )
    return sorted(terms)


def run_scoring():
    terms = load_unique_terms(INPUT_CSV)
    print(f"Loaded {len(terms)} unique terms from {INPUT_CSV}.")

    for cfg in MODELS:
        api_key = os.getenv(cfg["api_key"])
        if not api_key:
            print(f"Skipping {cfg['name']}: {cfg['api_key']} not set.")
            continue

        print(f"\n--- Scoring with {cfg['name']} ({cfg['model_id']}) ---")
        client = genai.Client(api_key=api_key)
        out_path = cfg["output_csv"]

        if os.path.exists(out_path):
            existing_df = pd.read_csv(out_path)
            # Keep only fully completed rows (drops any partial row from when it stopped)
            existing_df = existing_df.dropna(subset=["valence", "arousal", "dominance"])
            done_terms = set(existing_df["term"].astype(str))
            results = existing_df.to_dict("records")
            print(f"Resuming: {len(done_terms)} complete terms already scored.")
        else:
            done_terms = set()
            results = []

        remaining = [t for t in terms if t not in done_terms]
        print(f"Remaining terms to score: {len(remaining)}")

        try:
            for term in tqdm(remaining, desc=cfg["name"]):
                scores = query_scores(client, cfg["model_id"], term)
                results.append({"term": term, **scores})

                pd.DataFrame(results, columns=["term", "valence", "arousal", "dominance"]).to_csv(
                    out_path, index=False
                )
                time.sleep(cfg["sleep_sec"])
        except RuntimeError as e:
            print(f"\nStopped early: {e}")
        finally:
            client.close()

        print(f"Saved {len(results)} scored terms to {out_path}.")



In [40]:

if __name__ == "__main__":
    run_scoring()


Loaded 571 unique terms from responses_long.csv.

--- Scoring with gemini (gemini-3.5-flash-lite) ---
Resuming: 569 complete terms already scored.
Remaining terms to score: 2


gemini: 100%|██████████| 2/2 [00:09<00:00,  4.76s/it]

Saved 571 scored terms to vad_scores_gemini.csv.


# DEBUG

In [29]:
import os, pandas as pd
from google import genai
from google.genai import types

df = pd.read_csv("/Users/yukat/Developer/voice_eval/analysis/vad_scores_gemini.csv")
na_terms = df[df[["valence", "arousal", "dominance"]].isna().any(axis=1)]["term"].head(3).tolist()
print("Testing NA terms:", na_terms)

client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))
for t in na_terms:
    resp = client.models.generate_content(
        model="gemini-3.5-flash-lite",
        contents=PROMPT.format(term=t),
        config=types.GenerateContentConfig(
            temperature=0.0,
            max_output_tokens=128,
            response_mime_type="application/json",
        ),
    )
    print(f"\n--- {t} ---")
    print("RAW TEXT:", repr(resp.text))
client.close()


Testing NA terms: ['even', 'exuberant', 'fine']

--- even ---
RAW TEXT: '{\n  "valence": 0.000,\n  "arousal": -0.150,\n  "dominance": 0.050\n}'

--- exuberant ---
RAW TEXT: '[\n  {\n    "expression": "exuberant",\n    "valence": 0.850,\n    "arousal": 0.780,\n    "dominance": 0.650\n  }\n]'

--- fine ---
RAW TEXT: '{\n  "valence": 0.528,\n  "arousal": -0.057,\n  "dominance": 0.214\n}'


In [30]:
df = pd.read_csv("/Users/yukat/Developer/voice_eval/analysis/vad_scores_gemini.csv")
na_count = df[["valence", "arousal", "dominance"]].isna().any(axis=1).sum()
print(f"NA rows to re-score: {na_count}")


NA rows to re-score: 104


In [ ]:
pd.read_json("vad_na_raw_log.jsonl", lines=True)
